# Lab | Agent & Vector store

**Change the state union dataset and replicate this lab by updating the prompts accordingly.**

One such dataset is the [sonnets.txt](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) dataset or any other data of your choice from the same git.

# Combine agents and vector stores

This notebook covers how to combine agents and vector stores. The use case for this is that you've ingested your data into a vector store and want to interact with it in an agentic manner.

The recommended method for doing so is to create a `RetrievalQA` and then use that as a tool in the overall agent. Let's take a look at doing this below. You can do this with multiple different vector DBs, and use the agent as a way to route between them. There are two different ways of doing this - you can either let the agent use the vector stores as normal tools, or you can set `return_direct=True` to really just use the agent as a router.

## Create the vector store

In [ ]:
#!pip install chromadb langchain langchain_community langchain_openai langchain_text_splitters langchain-classic

In [6]:
from langchain_classic.chains import RetrievalQA  # Legacy chains
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [38]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [ ]:
# If you're using colab, run this
#os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

In [52]:
llm = OpenAI(temperature=0)

In [53]:
from pathlib import Path

relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break
#doc_path = str(Path(*relevant_parts) / "state_of_the_union.txt")
doc_path = str(Path(*relevant_parts) / "sonnets.txt")
print(doc_path)

c:\bianca\ironhack\_course_labs_projects\week7\lab-agent-vector-store\sonnets.txt


In [54]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()
#docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")
docsearch = Chroma.from_documents(texts, embeddings, collection_name="sonnets")


In [55]:
#state_of_union = RetrievalQA.from_chain_type(
sonnets = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [56]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"

In [17]:
from langchain_community.document_loaders import WebBaseLoader

In [57]:
#loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")
loader = WebBaseLoader("https://www.booking.com/tpi_faq.en.html/")

In [58]:
%pip install beautifulsoup4 lxml html5lib

Note: you may need to restart the kernel to use updated packages.


In [59]:
# Put near where bs4 is imported
try:
    from bs4 import BeautifulSoup
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "bs4 (beautifulsoup4) is required. Install with `pip install beautifulsoup4`"
    ) from e

In [68]:
docs = loader.load()
booking_texts = text_splitter.split_documents(docs)
booking_db = Chroma.from_documents(booking_texts, embeddings, collection_name="booking")
booking = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=booking_db.as_retriever()
)

## Create the Agent

In [69]:
# Import things that are needed generically
# Legacy agent imports
from langchain_classic.agents import AgentType, initialize_agent
from langchain_classic.tools import Tool  # or langchain.tools.Tool if using v1 Tool
from langchain_openai import OpenAI

In [70]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for when you need to answer questions related to the 'Sonnets' (by Shakespeare). Input should be a fully formed question.",
    ),
    Tool(
        name="Booking.com QA System",
        func=booking.run,
        description="useful for when you need to answer questions about Booking.com. Input should be a fully formed question.",
    ),
]

In [71]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [72]:
agent.invoke(
    "How do I book a room on Booking.com website?"
)



> Entering new AgentExecutor chain...
 I should use the Booking.com QA System to get the answer.
Action: Booking.com QA System
Action Input: "How do I book a room on Booking.com website?"
Observation:  To book a room on Booking.com, you can follow these steps:

1. Go to the Booking.com website.
2. Choose your preferred currency and language.
3. Enter your destination in the search bar and select your desired dates for check-in and check-out.
4. Select the number of adults and children (if any) who will be staying in the room.
5. Click on the "Search" button.
6. A list of available accommodations will appear. You can filter the results by price, location, facilities, and more.
7. Once you have found a suitable accommodation, click on the "Book now" button.
8. You will be asked to enter your personal details and payment information to confirm the booking.
9. After completing the booking process, you will receive a confirmation email with all the details of your reservation.
10. You can

{'input': 'How do I book a room on Booking.com website?',
 'output': 'To book a room on Booking.com, you can follow the steps provided by the Booking.com QA System.'}

In [74]:
agent.invoke("How does Shakespeare describe beauty in his sonnets?")



> Entering new AgentExecutor chain...
 I should use the Sonnets QA System to answer this question.
Action: Sonnets QA System
Action Input: "How does Shakespeare describe beauty in his sonnets?"
Observation:  Shakespeare describes beauty as something that is timeless and can be found in both the past and present. He also suggests that beauty is something that can be inherited and passed down through generations.
Thought: I now know the final answer.
Final Answer: Shakespeare describes beauty as timeless and inheritable in his sonnets.

> Finished chain.


{'input': 'How does Shakespeare describe beauty in his sonnets?',
 'output': 'Shakespeare describes beauty as timeless and inheritable in his sonnets.'}

## Use the Agent solely as a router

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [75]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for when you need to answer questions related to the 'Sonnets' (by Shakespeare). Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Booking QA System",
        func=booking.run,
        description="useful for when you need to answer questions about Booking.com. Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [77]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [78]:
agent.invoke("How do I book a room on Booking.com website?")



> Entering new AgentExecutor chain...
 First, I should think about what information I need to provide in order to book a room.
Action: Booking QA System
Action Input: "How do I book a room on Booking.com website?"
Observation:  To book a room on Booking.com, you can follow these steps:

1. Go to the Booking.com website.
2. Choose your preferred currency and language.
3. Enter your destination in the search bar and select your desired dates for check-in and check-out.
4. Select the number of adults and children (if any) who will be staying in the room.
5. Click on the "Search" button.
6. A list of available accommodations will appear. You can filter the results by price, location, facilities, and more.
7. Once you have found a suitable accommodation, click on the "Book now" button.
8. You will be asked to enter your personal details and payment information to confirm the booking.
9. After completing the booking process, you will receive a confirmation email with all the details of you

{'input': 'How do I book a room on Booking.com website?',
 'output': ' To book a room on Booking.com, you can follow these steps:\n\n1. Go to the Booking.com website.\n2. Choose your preferred currency and language.\n3. Enter your destination in the search bar and select your desired dates for check-in and check-out.\n4. Select the number of adults and children (if any) who will be staying in the room.\n5. Click on the "Search" button.\n6. A list of available accommodations will appear. You can filter the results by price, location, facilities, and more.\n7. Once you have found a suitable accommodation, click on the "Book now" button.\n8. You will be asked to enter your personal details and payment information to confirm the booking.\n9. After completing the booking process, you will receive a confirmation email with all the details of your reservation.\n10. You can also manage your booking and make changes online by signing in to your Booking.com account.'}

In [79]:
agent.invoke("How does Shakespeare describe beauty in his sonnets?")



> Entering new AgentExecutor chain...
 I should use the Sonnets QA System for this question.
Action: Sonnets QA System
Action Input: "How does Shakespeare describe beauty in his sonnets?"
Observation:  Shakespeare describes beauty as something that is timeless and can be found in both the past and present. He also suggests that beauty is something that can be inherited and passed down through generations.


> Finished chain.


{'input': 'How does Shakespeare describe beauty in his sonnets?',
 'output': ' Shakespeare describes beauty as something that is timeless and can be found in both the past and present. He also suggests that beauty is something that can be inherited and passed down through generations.'}

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [90]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Booking QA System",
        func=booking.run,
        description="useful for when you need to answer questions about Booking.com. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [91]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [92]:
agent.invoke("How can I book a room on Booking.com website? Did Shakespeare mention that in the sonnets?")



> Entering new AgentExecutor chain...
 I should check if Shakespeare mentioned anything about booking rooms in his sonnets.
Action: Sonnets QA System
Action Input: "Did Shakespeare mention booking rooms in his sonnets?"
Observation:  No, Shakespeare did not mention booking rooms in his sonnets.
Thought: I should use the Booking QA System to find out how to book a room on Booking.com.
Action: Booking QA System
Action Input: "How can I book a room on Booking.com website?"
Observation:  To book a room on Booking.com, you can follow these steps: 
1. Go to the Booking.com website 
2. Choose your preferred currency and language 
3. Enter your destination in the search bar 
4. Select your desired check-in and check-out dates 
5. Choose the number of adults and children staying 
6. Click on "Search" 
7. Browse through the available options and select the one that suits your needs 
8. Enter your personal and payment information 
9. Review your booking details and confirm your reservation 
10.

{'input': 'How can I book a room on Booking.com website? Did Shakespeare mention that in the sonnets?',
 'output': 'To book a room on Booking.com, you can follow the steps outlined by the Booking QA System.'}